# 02 · Baselines: could a fixed rule do it, and what is the bar?

*For learning. Nothing here runs in CI and nothing leaves this computer: the datasets registered by stage 1 on the local
MLflow server (`http://localhost:5050`, experiment `coach-retrieval-lab`), Ollama for two served models, and one model
already in the Hugging Face cache.*

**What this notebook settles.** Two things, both on the **dev** questions. (a) The vetting's first gate, *could a fixed rule
do it*: a plain word-matching table (TF-IDF) is scored against the embedding models, because if a free rule finds the
lesson as often as a model does, no model is worth training. (b) **Who the champion is and what the bar is**: the
off-the-shelf models as their makers ship them, scored the same way, so the fine-tune in stage 3 has a number to beat and a
sentence that says why that number.

**Which split it uses.** Dev only: `pairs_dev` (328 questions) and `honest_dev` (124 learner-worded paraphrases), read
back from stage 1's run by name and checked by digest. **Test** stays closed until stage 4. **Holdout** is locked until
the very end of the lab.

**Is this the experiment and the A/B test?** Half of it. Every setup below is one **eval run** in MLflow with its quality
(hit@1, recall@5, MRR, the weakest module), its speed (milliseconds per question, seconds to embed the course) and its
cost (model size now; Databricks prices at stage 8), all under one parent run, so the trade-off reads from one table. That
is the **offline** comparison, champion against challengers, on the same questions. The **online** A/B test, where real
learners' questions are split between two models and the measure is what they do next, is stage 7 and needs traffic this
computer does not have yet (0 typed questions recorded).

**The champion, said plainly.** Today's coach does *not* retrieve with embeddings: the page tells it which lesson the
learner is on. So the champion for the retrieval step is `bge-large` exactly as its makers ship it, with its search
prompt (recall@5 0.84 on 150 Check questions, PR #170). It is run here two ways, served by Ollama (the app's path) and
loaded locally with sentence-transformers (the way training will run), and the two are shown to be the same model. A
fine-tune has to earn its place where the lesson is *not* known: the study list, the blueprint's coach, Claude's tools
without a lesson.

**The lessons applied.** The section headings use the README's names for the habits. Where this worktree has a lesson
with that habit, its title is in brackets: *Measuring retrieval*, *Dev, test and holdout sets*, *Embedding models have
blind spots*, *Comparing on the same examples*, *Telling real gains from noise*, *How big should a vector be?*, *Report
each group, weighted by real traffic*. "Could a fixed rule do it" is the
vetting's first gate (`00_vet.ipynb`); "Real win or luck" and "Cost, speed, quality" are the README's names and not lesson
titles in this checkout.

**How to work through it.** Before each result there is a cell for **your guess**: write it down, then run the next cell.
Your guesses go into the MLflow run beside the real answers. The exercises have blanks (`"___"`); the answers are in
`02_baselines_answers.py`, which `DRILL_FILL_ANSWERS=1` uses to check the notebook runs end to end.

In [1]:
import importlib
import json
import os
import time
from collections import Counter
from pathlib import Path

os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT", "1")   # the run links are printed once, at the end

import mlflow
import pandas as pd

import retrieval as r
import tracking

FILL = os.environ.get("DRILL_FILL_ANSWERS") == "1"      # only for checking the notebook end to end
answers = importlib.import_module("02_baselines_answers")
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)


def attempt(value, name):
    # Your answer; if it still has a blank, a reminder (or, when checking the notebook end to end, the answer file's).
    if "___" not in str(value):
        return value
    if FILL:
        return getattr(answers, name)
    print("Fill in the ___ first, then run this cell again.")
    return None


tracking.connect()
stage1 = tracking.find_run(tracking.EXPERIMENT, "pairs · build and freeze", require=("pairs_dev.json", "honest_dev.json", "corpus.json"))
read_by_person = int(stage1.data.metrics.get("paraphrases_checked_by_hand", 0))   # stage 1's count: how much of the honest test a person has read
digests = {i.dataset.name: i.dataset.digest for i in stage1.inputs.dataset_inputs}


def load(name, targets="lesson"):
    # The rows stage 1 registered, read back from its run and registered again here: the digest must be the same one.
    rows = json.loads(Path(mlflow.artifacts.download_artifacts(run_id=stage1.info.run_id, artifact_path=f"{name}.json")).read_text())
    ds = tracking.table_dataset(rows, name, targets=targets)
    assert ds.digest == digests[name], f"{name}: digest {ds.digest} is not stage 1's {digests[name]}: the data moved, stop and look"
    return rows, ds


pairs_dev, ds_dev = load("pairs_dev")
honest_dev, ds_honest = load("honest_dev")
corpus, ds_corpus = load("corpus", targets=None)
splits = r.load_splits()
print(f"stage 1 run {tracking.short(stage1.info.run_id)}: pairs_dev {len(pairs_dev)} ({digests['pairs_dev']}), honest_dev {len(honest_dev)} "
      f"({digests['honest_dev']}), corpus {len(corpus)} paragraphs ({digests['corpus']}); content fingerprint {stage1.data.params['content_fingerprint']}; "
      f"paraphrases read by a person at stage 1: {read_by_person}")

stage 1 run 402d7435: pairs_dev 328 (bf0d1341), honest_dev 124 (70fd19b5), corpus 741 paragraphs (ad993ac3); content fingerprint 33910f9634f6; paraphrases read by a person at stage 1: 0


In [2]:
run = mlflow.start_run(run_name="baselines · champion and the bar (dev)",
                       description="(the full description is written at the end of the notebook, once the numbers are known)")
mlflow.set_tags({"kind": "coach_retrieval_lab", "lab.stage": "02_baselines", "git_commit": tracking.git_commit(),
                 "champion": r.CHAMPION.name, **{f"computer.{k}": str(v) for k, v in tracking.this_computer().items()}})
mlflow.log_params({"k": r.K, "seed": r.SEED, "champion": r.CHAMPION.name, "stage1_run_id": stage1.info.run_id,
                   "content_fingerprint": stage1.data.params["content_fingerprint"], "split": "dev",
                   "setups": " | ".join(s.name for s in r.BASELINE_SETUPS)})
for ds, context in ((ds_dev, "dev"), (ds_honest, "dev"), (ds_corpus, "corpus")):
    mlflow.log_input(ds, context=context)
tracking.guard_run()   # a cell that raises ends the run as FAILED, not as an empty FINISHED shell
you = tracking.Guesses()
print(f"MLflow run {tracking.short(run.info.run_id)} in experiment {run.info.experiment_id}; answers filled in: {FILL}")

MLflow run b8eedf08 in experiment 6; answers filled in: True


## 1. The measures and the five setups (*Measuring retrieval*)

> **In plain words.** A learner's question is turned into a list of numbers (a **vector**), every Learn paragraph was
> turned into one too, and the five paragraphs whose vectors point most nearly the same way are the search result.
>
> * **recall@5**: the share of questions where one of the right lesson's paragraphs is among the five closest. The main
>   number, because the coach reads five paragraphs.
> * **hit@1**: the stricter share where the *closest* paragraph is the right lesson's.
> * **MRR** (mean reciprocal rank): how high the first right paragraph sits, on average; 1.0 means always first, 0.5
>   means second on average.
> * **The Wilson range**: a share from n questions could have come out differently with n other questions; the range
>   says how far. 124 questions give a range about 15 points wide, 328 about 10.
> * **The weakest module**: the module where the setup found the fewest of its questions, with the count beside it, because
>   0 of 1 is not a finding.

Five setups, each a different way of turning text into vectors:

| setup | what it is | why it is here |
|---|---|---|
| `tf-idf words · fixed rule` | a word-count table: a question matches paragraphs that share its words | the fixed rule: if it does as well, no model is needed |
| `bge-large · local · with prompt` | **the champion**: bge-large loaded on this computer, with the makers' search prompt | the way training will run |
| `bge-large · ollama · with prompt` | the same model served by Ollama | the app's own path; the two must agree |
| `bge-large · local · no prompt` | the champion without its search prompt | the common mistake, measured |
| `embeddinggemma 768 · ollama · with prompt` | a different, smaller model with its own search prompts | a different maker's blind spots |

### Exercise 1: hit@1 and recall@5 by hand

Five questions. For each, the rank of the first paragraph from the right lesson: `[1, 3, None, 7, 1]` (`None`: the lesson
never came up). What are hit@1 and recall@5?

In [3]:
FIRST_RANKS = [1, 3, None, 7, 1]
HIT1_AND_RECALL5 = ("___", "___")   # two shares between 0 and 1, hit@1 first

In [4]:
mine = attempt(HIT1_AND_RECALL5, "HIT1_AND_RECALL5")
if mine is not None:
    hit1, recall5 = (float(x) for x in mine)
    fake = [{"id": str(i), "lesson": "x", "module": "m", "hit": bool(fr and fr <= 5), "first_rank": fr} for i, fr in enumerate(FIRST_RANKS)]
    s = r.summarise(fake)
    assert abs(hit1 - s["hit_at_1"]) < 0.01 and abs(recall5 - s["recall_at_5"]) < 0.01, \
        "Not quite: hit@1 counts the questions whose first right paragraph is at rank 1; recall@5 counts ranks 1 to 5. None never counts."
    print(f"Right: hit@1 {s['hit_at_1']:.1f} (two at rank 1), recall@5 {s['recall_at_5']:.1f} (ranks 1, 3 and 1; rank 7 is too far down). "
          f"The Wilson range for recall@5 on five questions is [{s['recall_at_5_low']:.2f}, {s['recall_at_5_high']:.2f}]: "
          f"five questions say almost nothing, which is why the tables below use 124 and 328.")

Right: hit@1 0.4 (two at rank 1), recall@5 0.6 (ranks 1, 3 and 1; rank 7 is too far down). The Wilson range for recall@5 on five questions is [0.23, 0.88]: five questions say almost nothing, which is why the tables below use 124 and 328.


## 2. Could a fixed rule do it? The easy questions (the vetting's first gate)

The **easy** questions are the 328 dev pairs as the course wrote them: 42 Check questions, 82 extra practice questions and
204 prepared chips (26 term chips, 174 option chips, 4 others). The known flaw: Check and practice questions were written
*from* the Learn text, so they share its words, and a word-matching table gets credit for that. Stage 1's smoke run saw
TF-IDF beat bge-large by two questions on a sample of 20.

**Your guesses first.** On these 328 questions, what recall@5 will the word table get? And the champion? (Does the rule
beat the embeddings on the easy questions?)

In [5]:
you.guess("tfidf_recall_easy", None)       # e.g. 0.6
you.guess("champion_recall_easy", None)    # e.g. 0.8

No guess written down. The habit only works if you commit to one before you look.
No guess written down. The habit only works if you commit to one before you look.


**How long this takes.** About a minute on this computer: the corpus (741 paragraphs) is embedded once per setup (about
10 s each for bge-large, 7 s for embeddinggemma, 1 s for the word table), then three sets of questions: all 328 easy ones,
the 124 Check and practice questions among them that stage 1 paraphrased (the **originals**), and their 124 paraphrases
(the **honest** set, section 3). Model loading is kept off the clock.

In [6]:
originals = {h["orig"] for h in honest_dev}
sets = {"easy": pairs_dev, "originals": [p for p in pairs_dev if p["id"] in originals], "honest": honest_dev}
print({name: len(qs) for name, qs in sets.items()}, "· easy by group:", dict(Counter(p["chip"] or p["source"] for p in pairs_dev)))
t0 = time.monotonic()
results = {s.name: r.evaluate_setup(s, corpus, sets, splits) for s in r.BASELINE_SETUPS}
seconds_all = time.monotonic() - t0
mlflow.log_metric("seconds_all_setups", round(seconds_all, 1))
print(f"{len(results)} setups × {sum(len(q) for q in sets.values())} questions in {seconds_all:.0f} s")

{'easy': 328, 'originals': 124, 'honest': 124} · easy by group: {'check': 42, 'more': 82, 'term': 26, 'option': 174, 'explain:gotcha': 4}


tf-idf words · fixed rule                corpus    0.8 s · easy recall@5 0.59 · originals recall@5 0.81 · honest recall@5 0.60 · 1 s in all


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/BAAI/bge-large-en-v1.5: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'BAAI/bge-large-en-v1.5'.


bge-large · local · with prompt          corpus    9.6 s · easy recall@5 0.63 · originals recall@5 0.85 · honest recall@5 0.73 · 16 s in all


bge-large · ollama · with prompt         corpus   10.1 s · easy recall@5 0.63 · originals recall@5 0.85 · honest recall@5 0.73 · 17 s in all


bge-large · local · no prompt            corpus    9.5 s · easy recall@5 0.60 · originals recall@5 0.81 · honest recall@5 0.69 · 12 s in all


embeddinggemma 768 · ollama · with prompt corpus    6.0 s · easy recall@5 0.65 · originals recall@5 0.87 · honest recall@5 0.81 · 10 s in all
5 setups × 576 questions in 56 s


In [7]:
easy = r.results_table(results, "easy")
you.reveal("tfidf_recall_easy", easy.loc[r.TFIDF.name, "recall@5"])
you.reveal("champion_recall_easy", easy.loc[r.CHAMPION.name, "recall@5"])
mlflow.log_table(easy.reset_index(), "tables/easy.json")
easy.round(3)

tfidf_recall_easy: it was 58.8% (no guess written down)
champion_recall_easy: it was 63.4% (no guess written down)


,n,recall@5,low,high,hit@1,hit@1 low,hit@1 high,mrr,weakest module,ms/question,s to embed course,model MB
setup,,,,,,,,,,,,
tf-idf words · fixed rule,328,0.588,0.534,0.640,0.363,0.313,0.416,0.468,foundations (0/3),0.01,0.82,NaN
bge-large · local · with prompt,328,0.634,0.581,0.684,0.436,0.383,0.490,0.527,security (3/13),4.51,9.60,1340.6
bge-large · ollama · with prompt,328,0.634,0.581,0.684,0.439,0.386,0.493,0.528,security (3/13),12.03,10.15,670.5
bge-large · local · no prompt,328,0.604,0.550,0.655,0.399,0.348,0.453,0.493,security (2/13),3.60,9.53,1340.6
embeddinggemma 768 · ollama · with prompt,328,0.646,0.593,0.696,0.448,0.395,0.502,0.543,experiments (15/30),6.91,5.97,621.9


**Reading it.** On all 328, every setup sits between 0.59 and 0.65 and the ranges overlap: the word table is *not* clearly
beaten. But this table is mostly chips (204 of 328), and the by-group table says where the numbers come from. Each cell is
the share with its Wilson 95% range in brackets, because a group of 26 says much less than a group of 174:

In [8]:
groups_easy = r.by_group_table(results, "easy")
mlflow.log_table(groups_easy.reset_index().rename(columns={"index": "setup"}), "tables/easy_by_group.json")
groups_easy

,check (n=42),more (n=82),term (n=26),option (n=174),explain:gotcha (n=4)
tf-idf words · fixed rule,"0.81 [0.67, 0.90]","0.82 [0.72, 0.89]","0.81 [0.62, 0.91]","0.40 [0.33, 0.48]","0.25 [0.05, 0.70]"
bge-large · local · with prompt,"0.88 [0.75, 0.95]","0.84 [0.75, 0.90]","0.81 [0.62, 0.91]","0.45 [0.38, 0.53]","0.50 [0.15, 0.85]"
bge-large · ollama · with prompt,"0.88 [0.75, 0.95]","0.84 [0.75, 0.90]","0.81 [0.62, 0.91]","0.45 [0.38, 0.53]","0.50 [0.15, 0.85]"
bge-large · local · no prompt,"0.81 [0.67, 0.90]","0.80 [0.71, 0.88]","0.85 [0.66, 0.94]","0.43 [0.35, 0.50]","0.50 [0.15, 0.85]"
embeddinggemma 768 · ollama · with prompt,"0.81 [0.67, 0.90]","0.90 [0.82, 0.95]","0.96 [0.81, 0.99]","0.44 [0.37, 0.52]","0.50 [0.15, 0.85]"


**Reading it, by group** (*Report each group, weighted by real traffic*). On the Check and practice questions, the ones
PR #170 measured, bge-large with its prompt is back at 0.84 to 0.88, which reproduces that result, and the word table is
close behind at about 0.81: the shared words carry it. The **option chips** ("why not *a timeout*?") are found less than
half the time by every setup, including the word table: they quote a wrong answer and name no topic, so no search can be
expected to find their lesson from the words alone. That is now a **finding across five setups**, not one model's blind
spot. *This is wrong if* learners type those words to the coach; today nobody does, they tap them beside the question.

**Decision for the bar:** option chips are reported as their own group and left **out** of the bar. A fine-tune that
"improved" on them would be learning to guess a lesson from a wrong answer, which is not the job. The term chips
(n = 26) stay in the picture but are too few to set a bar on: the champion's range on them runs from 0.62 to 0.91, nearly
30 points wide.

So, gate 1 on the easy questions: **not settled by them.** The word table is within a few points of every model on the
course's own wording. The question the gate really asks is the one on learner wording.

## 3. The honest test: the same questions in a learner's words (*Embedding models have blind spots*)

> **In plain words.** `honest_dev` is the 124 dev Check and practice questions rewritten by a local model (`qwen3:14b`) the
> way a junior developer would type them, without the course's terms. Stage 1 measured that the rewrites share fewer
> words with their lesson (overlap 0.54 → 0.37). A word table lives on shared words, so this is where it should fall; an
> embedding model is supposed to match *meaning*, so it should fall less. **The honest set is model-made**: no person has
> read all 124 rewrites yet, and at least one of the first three changed the meaning, so every number on it carries that
> caveat.

**Your guesses first.** On the 124 paraphrases, what recall@5 will the word table get? And the champion? (Does the rule
beat the embeddings on the paraphrases?)

In [9]:
you.guess("tfidf_recall_honest", None)       # e.g. 0.4
you.guess("champion_recall_honest", None)    # e.g. 0.7

No guess written down. The habit only works if you commit to one before you look.
No guess written down. The habit only works if you commit to one before you look.


In [10]:
honest = r.results_table(results, "honest")
you.reveal("tfidf_recall_honest", honest.loc[r.TFIDF.name, "recall@5"])
you.reveal("champion_recall_honest", honest.loc[r.CHAMPION.name, "recall@5"])
mlflow.log_table(honest.reset_index(), "tables/honest.json")
honest.round(3)

tfidf_recall_honest: it was 59.7% (no guess written down)
champion_recall_honest: it was 72.6% (no guess written down)


,n,recall@5,low,high,hit@1,hit@1 low,hit@1 high,mrr,weakest module,ms/question,s to embed course,model MB
setup,,,,,,,,,,,,
tf-idf words · fixed rule,124,0.597,0.509,0.679,0.315,0.239,0.401,0.441,models (0/3),0.02,0.82,NaN
bge-large · local · with prompt,124,0.726,0.641,0.797,0.476,0.390,0.563,0.590,foundations (0/1),6.59,9.60,1340.6
bge-large · ollama · with prompt,124,0.726,0.641,0.797,0.476,0.390,0.563,0.590,foundations (0/1),12.47,10.15,670.5
bge-large · local · no prompt,124,0.685,0.599,0.761,0.435,0.351,0.523,0.555,use-case (0/3),5.70,9.53,1340.6
embeddinggemma 768 · ollama · with prompt,124,0.815,0.737,0.873,0.548,0.461,0.633,0.661,platform-repos (1/3),7.40,5.97,621.9


The fair comparison is **the same 124 questions in two wordings**: the originals as the course wrote them, and their
paraphrases. The drop is what the wording alone costs each setup:

In [11]:
rows = []
for name, res in results.items():
    o, h = res["sets"]["originals"]["summary"], res["sets"]["honest"]["summary"]
    rows.append({"setup": name, "originals recall@5": o["recall_at_5"], "paraphrases recall@5": h["recall_at_5"],
                 "drop": o["recall_at_5"] - h["recall_at_5"], "originals hit@1": o["hit_at_1"], "paraphrases hit@1": h["hit_at_1"]})
two_wordings = pd.DataFrame(rows).set_index("setup")
mlflow.log_table(two_wordings.reset_index(), "tables/two_wordings.json")
mlflow.log_metrics({"champion.originals.recall_at_5": two_wordings.loc[r.CHAMPION.name, "originals recall@5"],
                    "champion.honest.recall_at_5": two_wordings.loc[r.CHAMPION.name, "paraphrases recall@5"],
                    "tfidf.drop_recall_at_5": two_wordings.loc[r.TFIDF.name, "drop"],
                    "champion.drop_recall_at_5": two_wordings.loc[r.CHAMPION.name, "drop"]})
two_wordings.round(3)

,originals recall@5,paraphrases recall@5,drop,originals hit@1,paraphrases hit@1
setup,,,,,
tf-idf words · fixed rule,0.815,0.597,0.218,0.605,0.315
bge-large · local · with prompt,0.855,0.726,0.129,0.694,0.476
bge-large · ollama · with prompt,0.855,0.726,0.129,0.694,0.476
bge-large · local · no prompt,0.806,0.685,0.121,0.645,0.435
embeddinggemma 768 · ollama · with prompt,0.871,0.815,0.056,0.710,0.548


**Reading it.** This is the table that settles gate 1. On the course's wording the word table and the models are a few
points apart; on the learner's wording the word table loses about 20 points and the embedding models lose less, so the
gap between rule and model opens up. **Gate 1: a fixed rule cannot do it on learner-worded questions**, and the honest
set is the test that shows it. Section 4 checks that the gap is not luck.

**The surprise** (a lead on dev, from 124 model-made paraphrases and one of several comparisons, to be confirmed on test at
stage 4; section 4 says how strong it is): `embeddinggemma`,
the smaller model, loses the least to the rewording and comes out *ahead* of the champion on the honest set, while tying
it on the easy set. *This is wrong if* the paraphrases happen to suit one model's training (both models were trained on
question-and-passage pairs, so no reason to expect that, but 124 questions is a small sample), or if a person's reading
removes rewrites that one model found and the other did not. It is a hunch that gemma's advantage comes from having seen
more everyday phrasing; nothing here tests that.

**The weakest module** in the tables is whichever module scored lowest, and a module with one dev question scores 0 or 1
by chance. So read it only where a module has at least five questions:

In [12]:
def weakest(name, set_name, at_least=5):
    return [f"{m['module']} {m['found']}/{m['n']}" for m in results[name]["sets"][set_name]["summary"]["by_module"] if m["n"] >= at_least][:3]

for set_name in ("easy", "honest"):
    print(f"{set_name:7} champion's three weakest modules with 5+ questions: {weakest(r.CHAMPION.name, set_name)}"
          f"   (the table's 'weakest': {results[r.CHAMPION.name]['sets'][set_name]['summary']['weakest_module']})")
mlflow.set_tags({"champion.weakest_5plus.easy": weakest(r.CHAMPION.name, "easy")[0], "champion.weakest_5plus.honest": weakest(r.CHAMPION.name, "honest")[0]})

easy    champion's three weakest modules with 5+ questions: ['security 3/13', 'llm-apps 4/10', 'models 5/11']   (the table's 'weakest': security)
honest  champion's three weakest modules with 5+ questions: ['security 2/6', 'production 2/6', 'rag 4/9']   (the table's 'weakest': foundations)


**Reading it.** Among modules with five or more questions, `security` is the champion's weakest on **both** sets (3 of 13
easy, 2 of 6 honest, tied with `production` on the honest set): the same module twice is more than noise, and it is the
first place stage 3 should look at the training pairs. The table's own "weakest" on the honest set is a module with one
question, which is what noise looks like.

## 4. Real win or luck: every setup paired against the champion (*Comparing on the same examples*)

> **In plain words.** Two setups answered the *same* questions, so compare them question by question. The questions both
> found and both missed say nothing about which is better; only the **disagreements** do. **McNemar's test** asks: if the
> two were really equal, how likely is a split this lopsided among the disagreements? A small p (under 0.05) says
> "unlikely to be luck". The **bootstrap range** of the gap redraws the 124 (or 328) questions with replacement 2,000
> times and keeps the middle 95% of the gaps: if the range straddles zero, the win is **not shown**. Two tests, two angles;
> when they disagree, trust the range, which is in the units the use case counts (questions found). A point that leans one
> way is not a gain until the range says so (*Telling real gains from noise*). And count the looks: with several setups on
> several sets, one "shown" among them is what luck alone produces fairly often, so a reading here is a lead for stage 4,
> where one pre-registered challenger-against-champion test is held to 0.05 on its own.

First, two checks on the champion itself.

**Served and local are the same model.** The app embeds through Ollama; training will use sentence-transformers. If the
two gave different vectors, a model trained one way would be served another. The **cosine** between each paragraph's two
vectors (1.0 means the two lists of numbers point exactly the same way; 0 means unrelated):

In [13]:
agree_docs = r.vector_agreement(results[r.CHAMPION.name]["doc"], results[r.CHAMPION_OLLAMA.name]["doc"])
agree_queries = r.vector_agreement(results[r.CHAMPION.name]["sets"]["easy"]["query"], results[r.CHAMPION_OLLAMA.name]["sets"]["easy"]["query"])
mlflow.log_metrics({"ollama_vs_local.docs.mean_cosine": agree_docs["mean_cosine"], "ollama_vs_local.docs.min_cosine": agree_docs["min_cosine"],
                    "ollama_vs_local.queries.mean_cosine": agree_queries["mean_cosine"], "ollama_vs_local.queries.min_cosine": agree_queries["min_cosine"]})
print(f"paragraphs: {agree_docs}\nquestions:  {agree_queries}")
print("Same model: the served copy (Ollama) and the local copy (sentence-transformers) point every vector the same way, "
      "so what stage 3 trains locally is what the app serves." if agree_docs["min_cosine"] > 0.99 else "LOOK: the two copies differ; stop and find out why before training.")

paragraphs: {'n': 741, 'mean_cosine': 1.0, 'min_cosine': 1.0}
questions:  {'n': 328, 'mean_cosine': 1.0, 'min_cosine': 0.9999}
Same model: the served copy (Ollama) and the local copy (sentence-transformers) point every vector the same way, so what stage 3 trains locally is what the app serves.


**The prompt is not optional.** bge-large's makers say to put "Represent this sentence for searching relevant passages: "
before each *question* (not before the paragraphs). Leaving it out is the common mistake. Here it is, paired against the
champion on both sets, with every other setup:

In [14]:
vs_easy = r.versus_champion(results, r.CHAMPION.name, "easy")
vs_honest = r.versus_champion(results, r.CHAMPION.name, "honest")
mlflow.log_table(vs_easy.reset_index(), "tables/versus_champion_easy.json")
mlflow.log_table(vs_honest.reset_index(), "tables/versus_champion_honest.json")
for name in vs_easy.index:
    key = name.replace(" ", "_").replace("·", "").replace("__", "_").replace("-", "_")
    mlflow.log_metrics({f"vs_champion.easy.gap.{key}": vs_easy.loc[name, "gap recall@5"], f"vs_champion.easy.p.{key}": vs_easy.loc[name, "McNemar p"],
                        f"vs_champion.honest.gap.{key}": vs_honest.loc[name, "gap recall@5"], f"vs_champion.honest.p.{key}": vs_honest.loc[name, "McNemar p"]})
print("easy (328 dev questions), gap = this setup minus the champion:")
display(vs_easy.round(3))
print("honest (124 paraphrases):")
display(vs_honest.round(3))
looks = vs_easy.attrs["comparisons"] + vs_honest.attrs["comparisons"]
chance = r.lucky_hit_chance(looks)
mlflow.log_params({"vs_champion.comparisons": looks, "vs_champion.lucky_hit_chance": chance})
print(f"\n{looks} real comparisons at the 5% level across both tables (the Ollama row is an identity check: 0 disagreements). "
      f"When nothing differs, at least one 'shown' turns up about {chance:.0%} of the time by luck alone, so a single 'shown' here is a lead "
      f"for stage 4, where one pre-registered test is held to 0.05 on its own. The reading column judges each row alone.")

easy (328 dev questions), gap = this setup minus the champion:


,only champion found,only this found,McNemar p,gap recall@5,gap low,gap high,reading
setup,,,,,,,
tf-idf words · fixed rule,43,28,0.096,-0.046,-0.098,0.003,not shown
bge-large · ollama · with prompt,0,0,1.000,0.000,0.000,0.000,not shown
bge-large · local · no prompt,15,5,0.041,-0.030,-0.058,-0.003,champion better
embeddinggemma 768 · ollama · with prompt,21,25,0.659,0.012,-0.030,0.052,not shown


honest (124 paraphrases):


,only champion found,only this found,McNemar p,gap recall@5,gap low,gap high,reading
setup,,,,,,,
tf-idf words · fixed rule,28,12,0.017,-0.129,-0.226,-0.032,champion better
bge-large · ollama · with prompt,0,0,1.000,0.000,0.000,0.000,not shown
bge-large · local · no prompt,12,7,0.359,-0.040,-0.105,0.024,not shown
embeddinggemma 768 · ollama · with prompt,7,18,0.043,0.089,0.016,0.161,other better



6 real comparisons at the 5% level across both tables (the Ollama row is an identity check: 0 disagreements). When nothing differs, at least one 'shown' turns up about 26% of the time by luck alone, so a single 'shown' here is a lead for stage 4, where one pre-registered test is held to 0.05 on its own. The reading column judges each row alone.


**Reading it.**

* *The word table against the champion.* On the easy questions the gap is small and its range reaches zero: **not shown**
  either way, which is the honest reading of "the rule keeps up on the course's own words". On the paraphrases the
  champion finds far more of the disagreements, McNemar agrees, and the range is below zero: **the champion's win over the
  rule is real** on learner wording.
* *Without the prompt.* On the easy set the champion is better and it is shown (the range stays below zero); on the honest
  set the same direction, but the range reaches zero, so with 124 questions it is **not shown**. The prompt is still not
  optional: it costs nothing and never hurt.
* *Served against local:* zero disagreements. The same model.
* *embeddinggemma against the champion.* Tied on the easy set (not shown either way). On the honest set it finds more of
  the disagreements (18 to 7), McNemar's p is 0.043 and the range sits just above zero (its low end is about two questions
  of 124). The reading column says "other better" because it judges the row alone; read it as **suggestive on dev**, not
  shown. It is one of six comparisons at the 5% level, and with six looks a result like this comes up about a quarter of
  the time when nothing differs; held to a stricter line for three challengers on one set (0.05 / 3, about 0.017) it would
  read "not shown". Dev is where choices are made, so this result has now been *looked at*; the test set at stage 4 is the
  only place it can be confirmed, and only if gemma is written into the bar as a challenger *before* test is opened.

### Exercise 2: what McNemar counts

Six questions, and whether the champion and the word table found each one's lesson. How many did *only* the champion
find, and how many *only* the word table?

In [15]:
caught = pd.DataFrame({"question": list("abcdef"),
                       "champion": [True, True, False, True, False, True],
                       "tfidf":    [True, False, False, True, True, False]})
ONLY_CHAMPION, ONLY_TFIDF = "___", "___"   # two counts

In [16]:
counts = attempt((ONLY_CHAMPION, ONLY_TFIDF), "ONLY_COUNTS")
if counts is not None:
    a, b = (int(x) for x in counts)
    got = r.paired([{"hit": x} for x in caught.champion], [{"hit": x} for x in caught.tfidf])
    assert (a, b) == (got["only_first"], got["only_second"]), "Not quite: only the rows where exactly one of them is True count. a and d (both) and c (neither) say nothing."
    print(f"Right: b and f for the champion, e for the word table; McNemar's p on 2 against 1 is {got['p_value']:.2f}: "
          f"three disagreements cannot show anything. The real tables above have 20 to 70 disagreements.")

Right: b and f for the champion, e for the word table; McNemar's p on 2 against 1 is 1.00: three disagreements cannot show anything. The real tables above have 20 to 70 disagreements.


### Exercise 3: reading a range

A challenger's gap over the champion comes out at −0.03 with a bootstrap range from −0.10 to +0.02. In the words of the
"reading" column, what is the verdict: `"other better"`, `"champion better"` or `"not shown"`?

In [17]:
READING = "___"   # one of the three readings

In [18]:
mine = attempt(READING, "READING")
if mine is not None:
    assert str(mine).strip().lower() == "not shown", "Not quite: a range that straddles zero cannot show a winner either way, whatever the point estimate says."
    print("Right: not shown. The point estimate leans to the champion, but with these questions the gap could as easily be +0.02. "
          "Say 'not shown', never 'no difference': the test may simply be too small to see one.")

Right: not shown. The point estimate leans to the champion, but with these questions the gap could as easily be +0.02. Say 'not shown', never 'no difference': the test may simply be too small to see one.


## 5. Cost, speed, quality: one table (*How big should a vector be?*)

> **In plain words.** Quality alone never decides. A learner waits for **milliseconds per question** (the time to turn
> their question into a vector; the search itself is a fraction of that). **Seconds to embed the course** is the one-off
> cost of indexing every paragraph, paid again whenever the lesson text or the model changes. **Model size** is memory on
> whatever serves it. The dollar lines are placeholders at 0 here, because on this computer everything is free; stage 8
> fills in Databricks' prices, where the champion is a pay-per-token endpoint (`databricks-bge-large-en`) with no upkeep,
> and any other model, gemma or a fine-tune, is a served model that costs by the hour.

Milliseconds per question move by a few from pass to pass (the GPU warms up, Ollama batches differently), so read them
as about-this-much, not exact. The two ms columns below are the same model timed on two sets in the same pass, and for the
champion they differ by about 2 ms, which is why the speed line of the bar is measured next as the median of several passes
on one set, with its spread.

In [19]:
csq = pd.DataFrame([{"setup": name,
                     "recall@5 honest": res["sets"]["honest"]["summary"]["recall_at_5"],
                     "recall@5 check+more": res["sets"]["originals"]["summary"]["recall_at_5"],
                     "ms/question (328)": res["sets"]["easy"]["speed_cost"]["ms_per_question"],
                     "ms/question (124)": res["sets"]["honest"]["speed_cost"]["ms_per_question"],
                     "s to embed course": res["sets"]["easy"]["speed_cost"]["seconds_to_embed_course"],
                     "model MB": res["sets"]["easy"]["speed_cost"]["model_size_mb"],
                     "usd here": res["sets"]["easy"]["speed_cost"]["usd_per_1k_questions"]} for name, res in results.items()]).set_index("setup")
mlflow.log_table(csq.reset_index(), "tables/cost_speed_quality.json")
csq.round(3)

,recall@5 honest,recall@5 check+more,ms/question (328),ms/question (124),s to embed course,model MB,usd here
setup,,,,,,,
tf-idf words · fixed rule,0.597,0.815,0.01,0.02,0.82,NaN,0.0
bge-large · local · with prompt,0.726,0.855,4.51,6.59,9.60,1340.6,0.0
bge-large · ollama · with prompt,0.726,0.855,12.03,12.47,10.15,670.5,0.0
bge-large · local · no prompt,0.685,0.806,3.60,5.70,9.53,1340.6,0.0
embeddinggemma 768 · ollama · with prompt,0.815,0.871,6.91,7.40,5.97,621.9,0.0


**Reading it.** The word table is effectively free and instant, and on learner wording it is 13 points behind: that is
the price of "free". bge-large locally in full precision is 1.3 GB; Ollama's copy is half that (it stores the weights in
half precision, which the agreement check above showed changes nothing that matters) and two to three times slower per question
through its HTTP path. embeddinggemma is the smallest file and embeds the course fastest, and on this pass it is the best
on the honest set: the rare case where cost, speed and quality point the same way. A fine-tune of bge-large would be the
same 1.3 GB and the same speed as the champion, so for it only the quality column can move.

### Exercise 4: the learner's wait

Which column of the table above is what a learner waits for when they ask the coach a question?

### The speed line, measured properly

One pass gives one number, and the table above shows the same champion at two different numbers. For the bar, speed is the
**median of several passes on the same questions** (the 124 course-worded originals, batch 32, model load off the clock),
and the **spread** (slowest pass minus fastest) is the smallest difference that can be told from noise on this computer. A
challenger is measured the same way, in the same session, and counts as no slower if its median is within the champion's
median plus that spread. Everything needed to repeat the measurement (which set, how many passes, the batch size) is logged.


In [20]:
speed = r.speed_passes(r.CHAMPION, [p["question"] for p in sets["originals"]], passes=5)
mlflow.log_metrics({f"speed.champion.{k}": v for k, v in speed.items() if isinstance(v, (int, float))})
mlflow.log_dict(speed, "speed_passes_champion.json")
print(f"{speed['setup']}: {speed['passes']} passes on {speed['n_texts']} questions (batch {speed['batch']}): "
      f"median {speed['ms_per_question_median']} ms, fastest {speed['ms_per_question_min']}, slowest {speed['ms_per_question_max']}, "
      f"spread {speed['ms_spread']} ms; each pass {speed['ms_each_pass']}")


bge-large · local · with prompt: 5 passes on 124 questions (batch 32): median 6.63 ms, fastest 6.62, slowest 6.91, spread 0.29 ms; each pass [6.91, 6.64, 6.63, 6.63, 6.62]


In [21]:
WAIT_COLUMN = "___"   # the column name, as written in the table

In [22]:
mine = attempt(WAIT_COLUMN, "WAIT_COLUMN")
if mine is not None:
    assert str(mine).replace(" ", "").lower().startswith("ms/question"), "Not quite: embedding the course happens once, before anyone asks; the learner waits for their own question to be embedded."
    print("Right: ms/question. Embedding the course is paid once, offline. Every setup here answers in under 15 ms, so on this "
          "computer speed does not separate them; the language model that writes the answer takes a thousand times longer.")

Right: ms/question. Embedding the course is paid once, offline. Every setup here answers in under 15 ms, so on this computer speed does not separate them; the language model that writes the answer takes a thousand times longer.


## 6. One eval run per setup (the record that counts)

Each setup becomes a child run under this notebook's parent run: its settings as params, every number from both question
sets as metrics (`recall_at_5`, `hit_at_1`, `mrr`, the per-module recalls, `ms_per_question`, `model_size_mb` for the easy
set; the same under `honest.` and `originals.`), the ranked rows as artifacts, and the three stage-1 datasets as its inputs,
so a run can be traced to the exact questions it answered.

In [23]:
honest_state = f"model-made by {r.PARAPHRASE_MODEL}, {read_by_person} of its rewrites read by a person (stage 1's count over dev and test)"
CAVEAT = ("Caveats: Check and practice questions share words with the Learn text, so the easy numbers flatter word matching; the honest set "
          f"(honest.*) is {honest_state}; option chips (174 of 328) name no topic and are reported "
          "as their own group, outside the bar.")
child_runs = {}
for setup in r.BASELINE_SETUPS:
    res = results[setup.name]
    e, o, h = res["sets"]["easy"], res["sets"]["originals"], res["sets"]["honest"]
    description = (f"**{setup.name}** on the dev split, one of five baseline setups under the parent run 'baselines · champion and the bar (dev)'. "
                   f"Main metrics: pairs_dev, {e['summary']['n']} questions (check, more and prepared chips). `originals.*`: the {o['summary']['n']} check and "
                   f"more questions that stage 1 paraphrased. `honest.*`: their {h['summary']['n']} learner-worded paraphrases. Corpus: {len(corpus)} Learn "
                   f"paragraphs embedded in {res['doc'].seconds} s; model load off the clock. "
                   + ("This is the champion: bge-large as its makers ship it, with its search prompt. " if setup is r.CHAMPION else "")
                   + CAVEAT)
    child = r.log_setup_run(setup, e["summary"], e["speed_cost"], [ds_dev, ds_honest, ds_corpus], description, run.info.run_id,
                            tags={"lab.stage": "02_baselines", "question_set": "pairs_dev", "role": "champion" if setup is r.CHAMPION else "baseline"},
                            rows=e["rows"], run_name=f"{setup.name} · dev")
    r.log_extra_set(child, "honest", h["summary"], h["speed_cost"], h["rows"])
    r.log_extra_set(child, "originals", o["summary"], o["speed_cost"], o["rows"])
    child_runs[setup.name] = child
    print(f"{setup.name:42} run {tracking.short(child)}")
mlflow.log_dict({"easy": {n: res["sets"]["easy"]["summary"] for n, res in results.items()},
                 "honest": {n: res["sets"]["honest"]["summary"] for n, res in results.items()},
                 "originals": {n: res["sets"]["originals"]["summary"] for n, res in results.items()}}, "summaries.json")
mlflow.log_dict({"ollama_vs_local": {"docs": agree_docs, "queries": agree_queries}, "child_runs": child_runs,
                 "originals_ids": sorted(originals)}, "details.json")

tf-idf words · fixed rule                  run 0b243e2e


bge-large · local · with prompt            run a5ee15c8


bge-large · ollama · with prompt           run be2f582c


bge-large · local · no prompt              run 0d5b3239


embeddinggemma 768 · ollama · with prompt  run b9f1d663


## 7. The bar, and what this means for the fine-tune (*Dev, test and holdout sets*)

A bar is a sentence with a reason, never a round number. It is set here on **dev**, where choices may be made; the number
the fine-tune is held to is the **test** set's at stage 4, measured once; and the **holdout** says the final word at the
end of the lab.

In [24]:
ch_h, ch_o = results[r.CHAMPION.name]["sets"]["honest"]["summary"], results[r.CHAMPION.name]["sets"]["originals"]["summary"]
headroom = 1 - ch_h["recall_at_5"]
the_bar = (f"A challenger earns the champion alias only if, on learner-worded questions about lessons it never trained on, its recall@5 beats "
           f"bge-large with its prompt (on dev: {ch_h['recall_at_5']:.2f}, Wilson range {ch_h['recall_at_5_low']:.2f} to {ch_h['recall_at_5_high']:.2f}, "
           f"n {ch_h['n']}; the test number is set at stage 4) with the paired bootstrap range of the gap above zero and McNemar's p under 0.05; "
           f"and, on the same course-worded Check and practice questions scored for both models (the forgetting check; champion on dev "
           f"{ch_o['recall_at_5']:.2f}, n {ch_o['n']}), the paired bootstrap range of (challenger minus champion) includes zero or sits above it and "
           f"McNemar does not show a champion win; and its median milliseconds per question over {speed['passes']} passes on the same questions, "
           f"measured in the same session with the same batch size and model load off the clock, is no higher than the champion's median plus the "
           f"champion's pass-to-pass spread (this pass: {speed['ms_per_question_median']} + {speed['ms_spread']} ms on {speed['n_texts']} questions). "
           f"Why this bar: on Databricks the champion is a pay-per-token endpoint with no upkeep and it already finds most course-worded questions, "
           f"so only a win on the hard questions that cannot be luck pays for serving a custom model; the forgetting check is paired for the same "
           f"reason the win is, because a challenger inside the champion's Wilson range could still have lost nine questions the champion found "
           f"and none the other way, which McNemar calls a real loss; and a speed difference smaller than the pass-to-pass noise cannot be told "
           f"from the champion, while a bigger one is a real slowdown. If the base model changes (embeddinggemma), the speed line is re-measured for that base.")
mlflow.set_tag("the_bar", the_bar)
mlflow.log_metrics({"bar.champion.honest.recall_at_5": ch_h["recall_at_5"], "bar.champion.honest.low": ch_h["recall_at_5_low"],
                    "bar.champion.honest.high": ch_h["recall_at_5_high"], "bar.champion.originals.recall_at_5": ch_o["recall_at_5"],
                    "bar.champion.originals.low": ch_o["recall_at_5_low"], "bar.champion.originals.high": ch_o["recall_at_5_high"],
                    "bar.champion.ms_per_question_median": speed["ms_per_question_median"], "bar.champion.ms_spread": speed["ms_spread"],
                    "bar.headroom.honest": headroom})
print("THE BAR.", the_bar)
found_o, lost = round(ch_o["recall_at_5"] * ch_o["n"]), 9
print(f"\nWhy the forgetting check is paired: the champion's Wilson range on the originals is {ch_o['recall_at_5_low']:.2f} to "
      f"{ch_o['recall_at_5_high']:.2f}. A challenger at {found_o - lost}/{ch_o['n']} = {(found_o - lost) / ch_o['n']:.2f} sits inside it, yet if those "
      f"{lost} are all questions only the champion found, McNemar's p is {tracking.mcnemar(lost, 0):.3f}: a real loss that a range check would wave through.")
print(f"\nHeadroom: the champion misses {headroom:.0%} of the honest dev questions ({ch_h['n'] - round(ch_h['recall_at_5'] * ch_h['n'])} of {ch_h['n']}); "
      f"that is the most any challenger can gain here.")

THE BAR. A challenger earns the champion alias only if, on learner-worded questions about lessons it never trained on, its recall@5 beats bge-large with its prompt (on dev: 0.73, Wilson range 0.64 to 0.80, n 124; the test number is set at stage 4) with the paired bootstrap range of the gap above zero and McNemar's p under 0.05; and, on the same course-worded Check and practice questions scored for both models (the forgetting check; champion on dev 0.85, n 124), the paired bootstrap range of (challenger minus champion) includes zero or sits above it and McNemar does not show a champion win; and its median milliseconds per question over 5 passes on the same questions, measured in the same session with the same batch size and model load off the clock, is no higher than the champion's median plus the champion's pass-to-pass spread (this pass: 6.63 + 0.29 ms on 124 questions). Why this bar: on Databricks the champion is a pay-per-token endpoint with no upkeep and it already finds most cours

**What this means for the fine-tune.**

1. **Gate 1 is passed, on learner wording only.** A word table keeps up on the course's own words and falls about 20
   points on a learner's. The fine-tune's job is the learner's words; the easy set is only the forgetting check.
2. **The headroom is about a quarter of the honest questions.** Of those, `security` is the module to read first: the
   champion's weakest on both sets. Hard negatives from the neighbouring lessons in that module are where stage 3 should
   start.
3. **A free challenger looks ahead on dev: suggestive, not shown.** `embeddinggemma`, as shipped, finds more of the honest
   disagreements than the champion (18 to 7, p 0.043), at half the size and the fastest course embedding. But that is one of
   six comparisons made here, and one such result is what luck produces about a quarter of the time, so it is a lead, not a
   passed check. Databricks' own retrieval guide says to try a stronger or different base model *before* fine-tuning; this
   is the first look at that, and worth following: if gemma is to be a challenger, write it into the bar now, before test is
   opened, and stage 3 then either fine-tunes bge-large and is held to gemma's test number too, or fine-tunes on top of
   gemma, because a fine-tune that loses to a free off-the-shelf model is not worth serving. On Databricks, mind, gemma is
   not free: it is a served model with the same upkeep a fine-tune would have, while bge-large is pay-per-token. So the cost
   column is not settled until stage 8.
4. **Everything above is on 124 model-made paraphrases.** The first thing that would move these numbers is a person
   reading the rewrites (section 4 of `01_pairs.ipynb`): if the ones gemma found and bge missed turn out to be the ones
   whose meaning changed, the surprise shrinks.

**Three open questions for the owner.**

1. *Which base model goes into stage 3:* bge-large (the plan, and the pay-per-token endpoint on Databricks) or
   embeddinggemma (ahead on dev, but only suggestively; cheaper on this computer, but a served model on Databricks either
   way)? Or both, as two challengers under the same bar? Whichever it is goes into the bar before test is opened.
2. *Do the option chips stay out of the bar for good?* This notebook leaves them out because nobody types them. If the
   study list or the blueprint's coach ever offers them as typed questions, they come back in as their own group.
3. *Who reads the 124 dev paraphrases, and when?* Until then every honest number says "model-made". The 20-item hand check
   in `01_pairs.ipynb` is the start; the test set's 84 need reading before stage 4, as the vetting's quality bar says.

In [25]:
tf_h, ge_h = results[r.TFIDF.name]["sets"]["honest"]["summary"], results[r.GEMMA.name]["sets"]["honest"]["summary"]
description = f"""**Stage 2 of the coach-retrieval lab: the baselines, the champion and the bar, on the dev split.** Built by `02_baselines.ipynb`; one child run per setup holds its numbers and the stage-1 datasets as inputs (pairs_dev {digests['pairs_dev']}, honest_dev {digests['honest_dev']}, corpus {digests['corpus']}; stage-1 run {tracking.short(stage1.info.run_id)}; content fingerprint {stage1.data.params['content_fingerprint']}).

**What.** Five setups ({' | '.join(s.name for s in r.BASELINE_SETUPS)}) scored on three dev question sets: all {len(pairs_dev)} dev pairs (the easy set: {dict(Counter(p['chip'] or p['source'] for p in pairs_dev))}), the {len(sets['originals'])} check and more questions among them that stage 1 paraphrased (originals), and their {len(honest_dev)} learner-worded paraphrases (the honest set, {honest_state}). Corpus {len(corpus)} Learn paragraphs, always searched whole. Measures: recall@5 and hit@1 with Wilson 95% ranges, MRR, recall by module (weakest first), ms per question, seconds to embed the course, model size; every setup paired against the champion with McNemar's test and a 2,000-draw bootstrap range of the gap.

**Why.** (a) The vetting's first gate: could a word-matching rule do it? (b) Who the champion is and what bar the fine-tune must clear. Today's coach does not retrieve with embeddings (the page tells it the lesson), so the champion for the retrieval step is bge-large as shipped, with its search prompt; a fine-tune earns its place only where the lesson is not known.

**Numbers (this pass).** Easy set recall@5: word table {easy.loc[r.TFIDF.name, 'recall@5']:.3f}, champion {easy.loc[r.CHAMPION.name, 'recall@5']:.3f}, gemma {easy.loc[r.GEMMA.name, 'recall@5']:.3f} (all ranges overlap; 174 option chips found under 0.5 by every setup). Same 124 questions, course wording -> learner wording: word table {two_wordings.loc[r.TFIDF.name, 'originals recall@5']:.3f} -> {tf_h['recall_at_5']:.3f}, champion {two_wordings.loc[r.CHAMPION.name, 'originals recall@5']:.3f} -> {ch_h['recall_at_5']:.3f} [{ch_h['recall_at_5_low']:.2f}, {ch_h['recall_at_5_high']:.2f}], gemma {two_wordings.loc[r.GEMMA.name, 'originals recall@5']:.3f} -> {ge_h['recall_at_5']:.3f} [{ge_h['recall_at_5_low']:.2f}, {ge_h['recall_at_5_high']:.2f}]. Paired against the champion on the honest set: word table gap {vs_honest.loc[r.TFIDF.name, 'gap recall@5']:+.3f} [{vs_honest.loc[r.TFIDF.name, 'gap low']:+.3f}, {vs_honest.loc[r.TFIDF.name, 'gap high']:+.3f}], p {vs_honest.loc[r.TFIDF.name, 'McNemar p']:.3f} ({vs_honest.loc[r.TFIDF.name, 'reading']}); no prompt {vs_honest.loc[r.NO_PROMPT.name, 'gap recall@5']:+.3f} [{vs_honest.loc[r.NO_PROMPT.name, 'gap low']:+.3f}, {vs_honest.loc[r.NO_PROMPT.name, 'gap high']:+.3f}] ({vs_honest.loc[r.NO_PROMPT.name, 'reading']}); gemma {vs_honest.loc[r.GEMMA.name, 'gap recall@5']:+.3f} [{vs_honest.loc[r.GEMMA.name, 'gap low']:+.3f}, {vs_honest.loc[r.GEMMA.name, 'gap high']:+.3f}], p {vs_honest.loc[r.GEMMA.name, 'McNemar p']:.3f} ({vs_honest.loc[r.GEMMA.name, 'reading']}); served vs local bge-large: 0 disagreements, min cosine {agree_docs['min_cosine']}. Champion's weakest module with 5+ questions: {weakest(r.CHAMPION.name, 'easy')[0]} (easy) / {weakest(r.CHAMPION.name, 'honest')[0]} (honest); the raw table's weakest on honest is a 1-question module. Speed: champion {speed['ms_per_question_median']} ms per question locally (median of {speed['passes']} passes on {speed['n_texts']} questions, batch {speed['batch']}, spread {speed['ms_spread']} ms; the one-pass numbers were {results[r.CHAMPION.name]['sets']['easy']['speed_cost']['ms_per_question']} on 328 and {results[r.CHAMPION.name]['sets']['honest']['speed_cost']['ms_per_question']} on 124), {results[r.CHAMPION_OLLAMA.name]['sets']['easy']['speed_cost']['ms_per_question']} through Ollama (one pass); sizes {results[r.CHAMPION.name]['sets']['easy']['speed_cost']['model_size_mb']} / {results[r.CHAMPION_OLLAMA.name]['sets']['easy']['speed_cost']['model_size_mb']} / {results[r.GEMMA.name]['sets']['easy']['speed_cost']['model_size_mb']} MB (local fp32, Ollama bge, Ollama gemma). All setups in {seconds_all:.0f} s.

**The bar.** {the_bar}

**Findings and their "wrong if".** Gate 1: a fixed rule keeps up on the course's wording and loses on a learner's; wrong if real typed questions turn out to reuse the course's words. Option chips are a different population for every setup (finding across five setups); wrong if learners type them. embeddinggemma is ahead of the champion on the honest dev set (18 to 7 of the disagreements, p {vs_honest.loc[r.GEMMA.name, 'McNemar p']:.3f}, gap low {vs_honest.loc[r.GEMMA.name, 'gap low']:+.3f}): suggestive on dev, one of {looks} comparisons at the 5% level (chance of one lucky 'shown' among them {chance:.0%}), to confirm on test at stage 4 as a challenger written into the bar beforehand; wrong if the model-made paraphrases suit one model, or if a person's reading removes the rewrites it won on. The security module is the champion's weakest on both sets (small counts: 13 and 6).

**Caveats.** (1) The honest set is {honest_state}; stage 1 flagged one meaning change in its first three rewrites. (2) Check and practice questions share words with the Learn text, so the easy numbers flatter word matching. (3) Dev has been looked at: these numbers chose nothing yet, but the test set is the only clean number from here. (4) Speed in the tables is one pass on this computer and moves by a few ms; the bar's speed line is the median of {speed['passes']} passes with its spread (speed_passes_champion.json). (5) Dollar costs are 0 here by construction; Databricks prices at stage 8, where bge-large is pay-per-token and any other model is served by the hour."""
mlflow.set_tag("mlflow.note.content", description)
print(description)

**Stage 2 of the coach-retrieval lab: the baselines, the champion and the bar, on the dev split.** Built by `02_baselines.ipynb`; one child run per setup holds its numbers and the stage-1 datasets as inputs (pairs_dev bf0d1341, honest_dev 70fd19b5, corpus ad993ac3; stage-1 run 402d7435; content fingerprint 33910f9634f6).

**What.** Five setups (tf-idf words · fixed rule | bge-large · local · with prompt | bge-large · ollama · with prompt | bge-large · local · no prompt | embeddinggemma 768 · ollama · with prompt) scored on three dev question sets: all 328 dev pairs (the easy set: {'check': 42, 'more': 82, 'term': 26, 'option': 174, 'explain:gotcha': 4}), the 124 check and more questions among them that stage 1 paraphrased (originals), and their 124 learner-worded paraphrases (the honest set, model-made by qwen3:14b, 0 of its rewrites read by a person (stage 1's count over dev and test)). Corpus 741 Learn paragraphs, always searched whole. Measures: recall@5 and hit@1 with Wilson 95% ra

In [26]:
mlflow.end_run()
print(f"This pass is saved: {tracking.run_link(run)}")

This pass is saved: http://localhost:5050/#/experiments/6 · run "baselines · champion and the bar (dev)" (b8eedf08)


**What comes next.** `03_train.ipynb` fine-tunes on the `pairs_train` lessons with hard negatives from the neighbouring
lessons, watches dev while it trains, and is held to the bar above. Before it, the owner answers the three questions:
which base model, whether the option chips stay out, and who reads the paraphrases.